# Explainable Machine Learning for Gallstone Disease

## 1. Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, validation_curve, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score, precision_score, confusion_matrix, classification_report, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

### 1.1 Dataset

We want to flip the labels for who HAS gallstone disease and who DOES NOT have gallstone disease. For future reference: patients with Gallstone Status == 1 has Gallstone Disease and patients with Gallstone Status == 0 does not have Gallstone Disease. 

In [ ]:
gd_df = pd.read_csv("dataset-uci.csv")

gd_df["Gallstone Status"] = 1 - gd_df["Gallstone Status"]
gd_df

In [ ]:
print((gd_df[gd_df["Gallstone Status"] == 1]).count())

## 2. Data Exploration

### 2.1 Initial Data Review

In [ ]:
gd_df.head(10).T

In [ ]:
gd_df.info()

In [ ]:
gd_df.shape

In [ ]:
gd_df.isnull().sum()

$$\boxed{\text{No null values found}}$$

In [ ]:
gd_df.describe().T

In [ ]:
gd_df.describe().T.sort_values("std", ascending=False)

In [ ]:
gd_df.duplicated().sum()

### 2.2 Cleaning

We want to inspect potential outliers from some medical features:

In [ ]:
#Medical ranges (these are limits we find reasonable)
medical_ranges = {
    "Age": (18, 100),
    "Weight": (30, 200),
    "High Density Lipoprotein (HDL)": (20, 100), 
    "Low Density Lipoprotein (LDL)": (50, 300), 
    "Body Mass Index (BMI)": (15, 50),
    "Creatinine": (0.5, 5.0), 
    "Glucose": (50, 400),  
    "Vitamin D": (2, 150),
    "Hemoglobin (HGB)": (8, 20),
    "Total Cholesterol (TC)": (100, 400), 
    "Triglyceride": (30, 1000),  
    "Obesity (%)": (0, 200),
    "C-Reactive Protein (CRP)": (0, 400),  
}

flagged = []
for feature, (low, high) in medical_ranges.items():
    outside = gd_df[(gd_df[feature] < low) | (gd_df[feature] > high)]
    for row, value in outside[feature].items():
        flagged.append({"row": row, "feature": feature, "value": value, "allowed": f"{low} to {high}"})
flagged = pd.DataFrame(flagged)

print(f"{len(flagged)} values outside reasonable range")
flagged

After analyzing the dataframe above, as well as dataset statistics, we notice that the "Obesity %" of patient 239 likely is incorrect. When it comes to the other rows from the table above, none of the values are as extreme and obvious errors. Therefore we have decided to keep an extra eye on these features, but will not remove anything per now.  

In [ ]:
gd_df[["Obesity (%)", "Height", "Weight"]].sort_values("Obesity (%)", ascending=False).head() # anomaly ? 

In [ ]:
gd_df[["Triglyceride", "Height", "Weight"]].sort_values("Triglyceride", ascending=False).head() # anomaly ? 

We notice that patient number 242 has suspicious trigliceride levels, especially when compared to the other patients in the triglycerid top 5. However, as we are not medical professionals, and this is not a obvious error, such as patient 239 with an Obesity % of 1954. We therefore concluded with keeping all of the Triglyceride rows. 

### 2.3 Visualization

In [ ]:
plt.figure(figsize=(12, 6))
plt.title("Patients With and Without Gallstone Disease, by Sex(n=319)")
ax = sns.countplot(x = "Gallstone Status", hue="Gender", data = gd_df, palette= ("lightblue", "pink"))
ax.set_xticklabels(["Absent", "Present"])

for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height())}",
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.xlabel("Gallstone disease")
plt.ylabel("Count")
plt.tight_layout()
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.legend(title="Gender", loc="upper right", labels=["Male", "Female"])
plt.show()

In [ ]:
#Age distribution of patients
plt.figure(figsize=(8, 5))
plt.hist(gd_df["Age"], bins=range(20, 101, 5), edgecolor="black", color="lightblue")
plt.title("Age distribution of patients (n=319)")
plt.xlabel("Age (years)")
plt.ylabel("Number of patients")
plt.grid(axis="y", linestyle="--", linewidth=0.5)
plt.show()

#Weight distribution patients
plt.figure(figsize=(8, 5))
plt.hist(gd_df["Weight"], bins=range(20, 101, 5), edgecolor="black", color="red")
plt.title("Weight distribution of patients (n=319)")
plt.xlabel("Weight (kg)")
plt.ylabel("Number of patients")
plt.grid(axis="y", linestyle="--", linewidth=0.5)
plt.show()

In [ ]:
plt.figure(figsize=(16, 12))
sns.heatmap(gd_df.corr(), cmap="coolwarm", center=0, vmin=-1, vmax=1, linewidths=0.3)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

In [ ]:
gd_df[["Gallstone Status", "C-Reactive Protein (CRP)"]].sort_values("C-Reactive Protein (CRP)", ascending=False).T

In [ ]:
has_gallstone = gd_df[gd_df["Gallstone Status"] == 1]
has_gallstone.describe().T

In [ ]:
no_gallstone = gd_df[gd_df["Gallstone Status"] == 0]
no_gallstone.describe().T

In [ ]:
gd_df.corrwith(gd_df["Gallstone Status"])

### 2.4 Preprocessing

Make decisions regarding the label, handle outliers, and train/test split

In [ ]:
X = gd_df.drop(["Gallstone Status"], axis = 1)
y = gd_df["Gallstone Status"]

#### Train/Test Split

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

### 2.5 Feature Selection

Look at high-correlation pairs, mutual information, drop table, ANOVA F-score, and decide which features to keep and which to drop.

In [ ]:
plt.figure(figsize=(16, 12))
sns.heatmap(X_train.corr(), cmap="coolwarm", center=0, vmin=-1, vmax=1, linewidths=0.3)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

In [ ]:
# correlation values between features
corr = X_train.corr().abs()

corr = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

high_corr = corr[(corr > 0.8) & (corr != 1)].stack()

high_corr.sort_values(ascending=False)

In [ ]:
X_train[["Total Body Fat Ratio (TBFR) (%)", "Lean Mass (LM) (%)"]].corr()

In [ ]:
mutual_info = mutual_info_classif(X_train, y_train, random_state=1)
mutual_info = pd.Series(mutual_info, index = X_train.columns).sort_values(ascending=False)
mutual_info

In [ ]:
drop_features = ["Total Body Water (TBW)", "Lean Mass (LM) (%)", "Obesity (%)"] # justification in feature_selection.md

X_train_selected = X_train.drop(columns=drop_features)
X_valid_selected = X_valid.drop(columns=drop_features)
X_test_selected = X_test.drop(columns=drop_features)

# now only 35 features
print(X_train_selected.shape)
print(X_test_selected.shape)

In [ ]:
SEED = 132

models = {
    "Logistic Regression" : Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED))
    ]),

    "Random Forest" : Pipeline([
        ("model", RandomForestClassifier(n_estimators=300, random_state=SEED))
    ]),

    "KNN" : Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),

    "SVM" : Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(probability=True, random_state=SEED))
    ]),

    "Gradient Boost" : Pipeline([
        ("model", GradientBoostingClassifier(n_estimators=300, random_state=SEED))
    ])
}

In [ ]:
results = []
roc_data = {}

for name, clf in models.items():
    clf.fit(X_train_selected, y_train)
    y_pred = clf.predict(X_valid_selected)
    y_prob = clf.predict_proba(X_valid_selected)[:,1]

    results.append({
        "model": name,
        "roc_auc": roc_auc_score(y_valid, y_prob),
        "accuracy": accuracy_score(y_valid, y_pred),
        "recall": recall_score(y_valid, y_pred),
        "precision": precision_score(y_valid, y_pred),
        "f1": f1_score(y_valid, y_pred)
    })

    fpr, tpr, _ = roc_curve(y_valid, y_prob)
    roc_data[name] = (fpr, tpr)

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
results_df

In [ ]:
plt.figure(figsize=(7,5))

for name, (fpr, tpr) in roc_data.items():
    plt.plot(fpr, tpr, label = name)
    
plt.plot([0,1],[0,1], linestyle="--", label="Random classifier")

plt.xlim(0, 1)
plt.ylim(0, 1)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC - Baseline Models (Validation)")
plt.legend()
plt.show()

In [ ]:
F_vals, p_vals = f_classif(X_train_selected, y_train)
feature_names = X_train_selected.columns

assert len(feature_names) == len(F_vals)

f_table = pd.DataFrame({
    "feature": feature_names,
    "F_score": F_vals,
    "p_value": p_vals
}).sort_values("F_score", ascending=False)

# ANOVA F-score Analysis Summary
print("="*60)
print("ANALYSIS OF VARIANCE (ANOVA) F-SCORE ANALYSIS")
print("="*60)
print(f"\nTotal features analyzed: {len(F_vals)}")
print(f"F-score range: {F_vals.min():.3f} to {F_vals.max():.3f}")
print(f"Mean F-score: {F_vals.mean():.3f}")
print(f"Median F-score: {np.median(F_vals):.3f}")
print(f"Std deviation: {F_vals.std():.3f}")
print(f"\nFeatures with F >= 1.0: {(F_vals >= 1.0).sum()}")
print(f"Features with F >= 0.70: {(F_vals >= 0.70).sum()}")

print(f"Features with p < 0.001: {(p_vals < 0.001).sum()} (extremely significant) (***)")
print(f"Features with p < 0.01: {(p_vals < 0.01).sum()} (highly significant) (**)")
print(f"Features with p < 0.05: {(p_vals < 0.05).sum()} (statistically significant) (*)")

print("\n" + "="*60)
print("TOP FEATURES BY ANOVA F-SCORE")
print("="*60)
for _, row in f_table.iterrows():
    significance = "***" if row['p_value'] < 0.001 else "**" if row['p_value'] < 0.01 else "*" if row['p_value'] < 0.05 else ""
    print(f"{row['feature']:<35} F={row['F_score']:>8.3f}  p={row['p_value']:.4e} {significance}")

print("\nSignificance: (***) p<0.001, (**) p<0.01, (*) p<0.05")

# Select features
selected_df = f_table[f_table["F_score"] >= 0.70]
selected_features = selected_df["feature"].tolist()

print(f"\nSelected {len(selected_features)} features (F >= 0.70):")
for i, row in enumerate(selected_df.itertuples(), 1):
    print(f"{i:2d}. {row.feature:<35} (F={row.F_score:.3f})")

# Transform the data
X_train_anova = X_train_selected[selected_features]    
X_valid_anova = X_valid_selected[selected_features]
X_test_anova = X_test[selected_features]

print(f"\nOriginal features: {X_train_selected.shape[1]}")
print(f"Selected features: {X_train_selected.shape[1]}")

# Compare baseline model performance with feature selection
print("\n" + "="*60)
print("Performance Comparison: All Features vs. Feature Selection")
print("="*60)

for name, clf in models.items():
    clf_all = clone(clf)
    clf_selected = clone(clf)

    clf_all.fit(X_train_selected, y_train)
    y_pred_all = clf_all.predict_proba(X_valid_selected)[: ,1]
    roc_auc_all = roc_auc_score(y_valid, y_pred_all)

    clf_selected.fit(X_train_anova, y_train)
    y_pred_selected = clf_selected.predict_proba(X_valid_anova)[: ,1]
    roc_auc_selected = roc_auc_score(y_valid, y_pred_selected)

    print(f"\n{name}:")
    print(f"  All features ({X_train_selected.shape[1]}):      ROC-AUC = {roc_auc_all:.4f}")
    print(f"  Selected features ({len(selected_features)}):  ROC-AUC = {roc_auc_selected:.4f}")
    print(f"  Difference:                {roc_auc_selected - roc_auc_all:+.4f}")

## 3. Model Selection

### 3.1 Research Question

Can machine learning models effectively predict gallstone
disease, and do certain features drive the model toward the
correct decision?

### 3.2 Model Justification

#### Logistic Regession 
Logistic Regression was selected as a baseline model because it is simple and easy to interpret. It estimates the probability of gallstone disease based on the input features, providing a useful comparison for more comlex models. 

#### Random Forest
Random Forest combines multiple dicision trees to make predictions. It was selected because it can capture nonlinear information between features. 

#### Gradient Boost
Gradient Boosting build decision trees(scikit-learn default) sequentially, where each new tree attempts to improve the predictions of the existing model. It was selected to investigate whether the approach could improve predictive performance compared to Random Forest and the simpler models. 

#### SVM (Support Vector Machine)
SVM tries to find the best dividing line that separates the different classes, with as much space between them as possible. It was selected because it can perform well on small datasets. The Radial Basis Function(RBF) kernel, wich is used by default in sckikit-learn, allows the model find nonlinear pattern between the features and gallstone disease. 

#### KNN (K-Nearest Neightbors)
KNN predicts a new observation based on its closest training observations. It was selected as a simple distance-based approach to investigate whether patients with similar characteristics also tend to have similar gallstone outcomes. 

## 4. Training

### 4.1 Data Preparation

Pipelines, cross-validation

In [ ]:
scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC-AUC": "roc_auc"
}

In [ ]:
cross_validation = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=SEED
)

### 4.2 Cross-validation Results

In [ ]:
results = {}

for name, model in models.items():
    if name in ["Logistic Regression", "Random Forest", "Gradient Boost"]:
        scores = cross_validate(
            model,
            X_train_anova,
            y_train,
            cv=cross_validation,
            scoring=scoring
        )
        results[name] = scores

### Results of 10-fold with 35 features

In [ ]:
comparison = pd.DataFrame()

for name, scores in results.items():
    for metric in scoring:
        mean = scores[f"test_{metric}"].mean()
        sd = scores[f"test_{metric}"].std()

        comparison.loc[name, metric] = f"{mean: .3f} ± {sd: .3f}"

display(comparison.round(3))

### 5.1 Metrics

We have decided to use the following metrics for evaluating machine learning model performance:

$$\text{Precision: } \quad \quad \quad \quad \frac{TP}{TP + FP}$$
$$\text{Recall: } \quad \quad \quad \quad \frac{TP}{TP + FN}$$
$$\text{F-measure: } \quad \quad \quad \quad \frac{2 \times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$
$$\text{Accuracy: } \quad \quad \quad \quad \frac{TP + TN}{FN + TP + TN +FP}$$



TP = True Positive, FP = False Positive, TN = True Negative, TP = True Positive

### 5.2 Parameter Tuning

In [ ]:
# Grid search function
def grid_search(model: str, param_grid: dict, score: str = "roc_auc"):
    '''
    Tune one model's hyperparameters with cross-validated grid search.

    Every combination in param_grid is tried with stratified 10-fold CV on the training set, scored by ROC-AUC. Prints the best parameters and their mean CV score.

    Parameters:
        model (str): Name of a model in the `models` dict, e.g. "Logistic Regression" or "KNN".
        param_grid (dict): Hyperparameters to try. Keys use the pipeline prefix, e.g. {"model__C": [0.01, 0.1, 1, 10]}.
        score (str): Metric used to rank the grid combinations, e.g. "roc_auc" or "recall". Default "roc_auc".
    '''

    grid_search = GridSearchCV(estimator=models[model],
                            param_grid=param_grid,
                            cv=cross_validation,
                            scoring=score,
                            return_train_score=True)

    grid_search.fit(X_train_anova, y_train)

    print("Best parameters: ", grid_search.best_params_)
    print("Best score: ", grid_search.best_score_)

    return grid_search

In [ ]:
param_grid= {
    "model__C" : [0.01, 0.1, 1, 10, 100]
    }

lr_best = grid_search("Logistic Regression", param_grid=param_grid)

In [ ]:
results = pd.DataFrame(lr_best.cv_results_)

plt.plot(
    results["param_model__C"].astype(float),
    results["mean_train_score"],
    marker="o",
    label="Training"
)

plt.plot(
    results["param_model__C"].astype(float),
    results["mean_test_score"],
    marker="o",
    label="Validation"
)

plt.xscale("log")
plt.xlabel("C")
plt.ylabel("ROC-AUC")
plt.title("Training vs Validation - Logistic Regression")
plt.legend()
plt.show()

For Logistic Regression we decided to tune the following parameter following the results of the GridsearchCV:

- C: 1 -> 10

This value of the hyperparameter achieved the best cross validation ROC-AUC (0.85433).

The tuning gives a slight improvement in performance, whilst we still keep some regularization.

In [ ]:
param_grid = {
    "model__n_estimators" : [100, 200],
    "model__max_depth" : [None, 4, 6, 8],
    "model__min_samples_leaf" : [1, 2, 4],
    "model__max_features": ["sqrt", "log2", 0.3, 0.5]
}

rf_best = GridSearchCV(estimator=models["Random Forest"],
                           param_grid=param_grid,
                           cv=cross_validation,
                           scoring="roc_auc",
                           return_train_score=True)

rf_best.fit(X_train_anova, y_train)


print("Best parameters: ", rf_best.best_params_)
print("Best score: ", rf_best.best_score_)

In [ ]:
results = pd.DataFrame(rf_best.cv_results_)

results = results[
    (results["param_model__min_samples_leaf"] == rf_best.best_params_["model__min_samples_leaf"]) &
    (results["param_model__max_features"] == rf_best.best_params_["model__max_features"]) &
    (results["param_model__n_estimators"] == rf_best.best_params_["model__n_estimators"])
]

results = results.sort_values("param_model__max_depth", na_position="last")

plt.plot(
    results["param_model__max_depth"].astype(str),
    results["mean_train_score"],
    marker="o",
    label="Training"
)

plt.plot(
    results["param_model__max_depth"].astype(str),
    results["mean_test_score"],
    marker="o",
    label="Validation"
)

plt.xlabel("max_depth")
plt.ylabel("ROC-AUC")
plt.title("Training vs Validation - Random Forest")
plt.legend()
plt.show()

For Random Forest we decided to tune the following parameters following the results of the GridsearchCV:

- max_depth:  None -> 6
- max_features: "sqrt" -> 0.5
- min_samples_leaf: 1 -> 1 # we keep as is
- n_estimators: 100 -> 100 # we keep as is

This combination of hyperparameters achieved the best cross validation ROC-AUC (0.87335).

We chose to visualize the training and validation ROC-AUC for max_depth given the fact it is an important factor in the complexity of the tree and can help illustrate overfitting. The other hyperparameters are fixed at their best values. 

The plot shows us that the validation performance peaks at max_depth = 6, while further increasing of depth doesnt improve the performance. 

We can also observe that the ROC-AUC for training is close to 1.0. This is considerably higher than the ROC-AUC for validation, indicating the model may be overfitting the training data. However, we chose the hyperparameters based on the validation performance in stead of the training performance. The validation results still suggest reasonable generalization, and we therefore continue with these hyperparameter values.

In [ ]:
param_grid = {
    "model__n_estimators" : [150, 300], # add back 400
    "model__learning_rate" : [0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0.01],
    "model__max_depth" : [2, 3, 6, 8]
}

gb_best = GridSearchCV(estimator=models["Gradient Boost"],
                           param_grid=param_grid,
                           cv=cross_validation,
                           scoring="roc_auc",
                           return_train_score=True)

gb_best.fit(X_train_anova, y_train)


print("Best parameters: ", gb_best.best_params_)
print("Best score: ", gb_best.best_score_)

In [ ]:
results = pd.DataFrame(gb_best.cv_results_)

results = results[
    (results["param_model__max_depth"] == gb_best.best_params_["model__max_depth"]) &
    (results["param_model__n_estimators"] == gb_best.best_params_["model__n_estimators"])
]

results = results.sort_values("param_model__learning_rate")

plt.plot(
    results["param_model__learning_rate"].astype(float),
    results["mean_train_score"],
    marker="o",
    label="Training"
)

plt.plot(
    results["param_model__learning_rate"].astype(float),
    results["mean_test_score"],
    marker="o",
    label="Validation"
)

plt.xscale("log")
plt.xlabel("learning_rate")
plt.ylabel("ROC-AUC")
plt.title("Training vs Validation - Gradient Boost")
plt.legend()
plt.show()

**Note: TODO**

For Gradiant Boost we decided to tune the following parameters following the results of the GridsearchCV:

- 

This combination of hyperparameters achieved the best cross validation ROC-AUC ().

> We have now picked out 3 models we want to try out with optimal parameters which are Logistic Regression with C = 1, Random Forest with {model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 4} and Gradient Boost with {'model__learning_rate': 0.1, 'model__max_depth': 2'}

### Logistic Regression

In [ ]:
model = lr_best.best_estimator_
y_pred = model.predict(X_test_anova)
y_prob = model.predict_proba(X_test_anova)[:, 1]

In [ ]:
print('Accuracy score: %.3f' % accuracy_score(y_test, y_pred))
print('F1-score: %.3f' % f1_score(y_test, y_pred))
print('Recall score: %.3f' % recall_score(y_test, y_pred))
print('Precision score: %.3f' % precision_score(y_test, y_pred))
print('ROC-AUC score: %.3f' % roc_auc_score(y_test, y_prob))

In [ ]:
cm = confusion_matrix(y_test, y_pred, normalize='true')

sns.heatmap(cm, 
            annot=True,
            fmt='g', 
            xticklabels=['Not Gallstone','Gallstone'],
            yticklabels=['Not Gallstone','Gallestone'])
plt.ylabel('Actual', fontsize=13)
plt.title('Confusion Matrix', fontsize=17, pad=20)
plt.gca().xaxis.set_label_position('top') 
plt.xlabel('Prediction', fontsize=13)
plt.gca().xaxis.tick_top()

plt.gca().figure.subplots_adjust(bottom=0.2)
plt.gca().figure.text(0.5, 0.05, 'Prediction', ha='center', fontsize=13)
plt.show()
print(classification_report(y_test, y_pred))

#### Random Forest

In [ ]:
model = rf_best.best_estimator_
y_pred = model.predict(X_test_anova)
y_prob = model.predict_proba(X_test_anova)[:, 1]

In [ ]:
print('Accuracy score: %.3f' % accuracy_score(y_test, y_pred))
print('F1-score: %.3f' % f1_score(y_test, y_pred))
print('Recall score: %.3f' % recall_score(y_test, y_pred))
print('Precision score: %.3f' % precision_score(y_test, y_pred))
print('ROC-AUC score: %.3f' % roc_auc_score(y_test, y_prob))

In [ ]:
cm = confusion_matrix(y_test, y_pred, normalize='true')

sns.heatmap(cm, 
            annot=True,
            fmt='g', 
            xticklabels=['Not Gallstone','Gallstone'],
            yticklabels=['Not Gallstone','Gallestone'])
plt.ylabel('Actual', fontsize=13)
plt.title('Confusion Matrix', fontsize=17, pad=20)
plt.gca().xaxis.set_label_position('top') 
plt.xlabel('Prediction', fontsize=13)
plt.gca().xaxis.tick_top()

plt.gca().figure.subplots_adjust(bottom=0.2)
plt.gca().figure.text(0.5, 0.05, 'Prediction', ha='center', fontsize=13)
plt.show()
print(classification_report(y_test, y_pred))

In [ ]:
model = gb_best.best_estimator_
y_pred = model.predict(X_test_anova)
y_prob = model.predict_proba(X_test_anova)[:, 1]

In [ ]:
print('Accuracy score: %.3f' % accuracy_score(y_test, y_pred))
print('F1-score: %.3f' % f1_score(y_test, y_pred))
print('Recall score: %.3f' % recall_score(y_test, y_pred))
print('Precision score: %.3f' % precision_score(y_test, y_pred))
print('ROC-AUC score: %.3f' % roc_auc_score(y_test, y_prob))

In [ ]:
cm = confusion_matrix(y_test, y_pred, normalize='true')

sns.heatmap(cm, 
            annot=True,
            fmt='g', 
            xticklabels=['Not Gallstone','Gallstone'],
            yticklabels=['Not Gallstone','Gallestone'])
plt.ylabel('Actual', fontsize=13)
plt.title('Confusion Matrix', fontsize=17, pad=20)
plt.gca().xaxis.set_label_position('top') 
plt.xlabel('Prediction', fontsize=13)
plt.gca().xaxis.tick_top()

plt.gca().figure.subplots_adjust(bottom=0.2)
plt.gca().figure.text(0.5, 0.05, 'Prediction', ha='center', fontsize=13)
plt.show()
print(classification_report(y_test, y_pred))

### 5.4 Explainability